# Demo 04 : A workflow or an agent

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import csv
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

db = sqlite3.connect(data("demo-world/service.db"), check_same_thread=False)
db.row_factory = sqlite3.Row

SECTIONS = []
for path in sorted(Path(data_folder("demo-world/manuals")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]
TICKETS = [", ".join(f"{k}={v}" for k, v in t.items())
           for t in csv.DictReader(open(data("demo-world/tickets.csv"), encoding="utf-8"))]


def rows(sql, *args):
    return [dict(r) for r in db.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]


print("sections:", len(SECTIONS), " tickets:", len(TICKETS),
      " buses:", len(rows("SELECT * FROM buses")), " parts:", len(rows("SELECT * FROM parts")))

sections: 82  tickets: 43  buses: 10  parts: 13


Four tools over the data, a meter for calls and tokens, `diagnose`, which
looks up the manual and the part in plain Python, and four reports.

In [3]:
@tool
def bus_info(fleet_no: str) -> str:
    """Model, depot and firmware of one bus, by fleet number such as CE12-2021-0117."""
    return str(rows("SELECT * FROM buses WHERE fleet_no = ?", fleet_no.strip().upper()) or "no such bus")


@tool
def part_stock(part_no: str) -> str:
    """Name, units in stock, price and depot of one spare part, by part number such as CE-CF-02."""
    return str(rows("SELECT * FROM parts WHERE part_no = ?", part_no.strip().upper()) or "no such part")


@tool
def parts_for_model(model: str) -> str:
    """Every spare part for a bus model such as CE-12, with units in stock."""
    return str(rows("SELECT * FROM parts WHERE model = ?", model.strip().upper()))


@tool
def search_manuals(query: str) -> str:
    """Search the service manuals and bulletins. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

def diagnose(fleet_no: str, error_code: str) -> dict:
    bus = rows("SELECT model, depot_id FROM buses WHERE fleet_no = ?", fleet_no)[0]
    manual = next(s for s in SECTIONS if s.startswith(f"# {bus['model']} error codes"))
    row = next(line for line in manual.splitlines() if line.startswith(f"| {error_code} |"))
    meaning, action = [cell.strip() for cell in row.split("|")[2:4]]
    part = next((p for p in rows("SELECT * FROM parts WHERE model = ?", bus["model"])
                 if p["name"] in action), None)
    return {"model": bus["model"], "meaning": meaning, "action": action, "part": part}

REPORTS = [
    "CE12-2021-0117 : F-CE-110 on the afternoon route.",
    "CD12-2024-0098 shows F-CD-202 again, pressure low.",
    "Bus RR14-2022-0210, error F-RR-120 at the last stop.",
    "The ramp on MM09-2025-2311 will not come out. No code on the display.",
]
NOTE = ("Write a three-line note for the mechanic : the likely cause, the first action, "
        "and the part with its stock.")

## a · workflow-or-agent

We fix the workflow's path in code. A regular expression finds the fleet
number and the error code, `diagnose` reads the manual and the parts table,
and the model writes the note : one model call per report. A report without a
code goes to second line.

In [4]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class Ticket(TypedDict, total=False):
    report: str
    fleet_no: str | None
    error_code: str | None
    facts: dict
    note: str


def parse(state: Ticket) -> dict:
    fleet = re.search(r"[A-Z]{2}\d{2}-\d{4}-\d{4}", state["report"])
    error = re.search(r"F-[A-Z]{2}-\d{3}", state["report"])
    return {"fleet_no": fleet and fleet.group(), "error_code": error and error.group()}


def route(state: Ticket) -> Literal["look_up", "escalate"]:
    return "look_up" if state["fleet_no"] and state["error_code"] else "escalate"


def look_up(state: Ticket) -> dict:
    return {"facts": diagnose(state["fleet_no"], state["error_code"])}


def write_note(state: Ticket) -> dict:
    return {"note": llm.invoke(f"{NOTE}\n\nReport : {state['report']}\nFacts : {state['facts']}").content}


builder = StateGraph(Ticket)
builder.add_node("parse", parse)
builder.add_node("look_up", look_up)
builder.add_node("write_note", write_note)
builder.add_node("escalate", lambda state: {"note": "No fleet number or error code : sent to second line."})
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("look_up", "write_note")
builder.add_edge("write_note", END)
builder.add_edge("escalate", END)
workflow = builder.compile()
print(workflow.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	look_up(look_up)
	write_note(write_note)
	escalate(escalate)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	look_up --> write_note;
	parse -.-> escalate;
	parse -.-> look_up;
	escalate --> __end__;
	write_note --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [5]:
runs = {}
for report in REPORTS:
    meter = Meter()
    note = workflow.invoke({"report": report}, {"callbacks": [meter]})["note"]
    runs[report] = {"workflow": meter}
    print(f"> {report}\n{note}\n[{meter}]\n")

> CE12-2021-0117 : F-CE-110 on the afternoon route.
Likely cause: Traction system over-temperature.
First action: Check the traction cooling fan; replace it if airflow is below 80%.
Part and stock: Traction cooling fan (CE-CF-02), 6 in stock at Rietpoort.
[1 calls, 145 input tokens, 56 output tokens]



> CD12-2024-0098 shows F-CD-202 again, pressure low.
Likely cause: Fuel pump pressure is low on CD-12 (F-CD-202).
First action: Withdraw the bus from service and call the diesel specialist; replacement requires a workshop bay.
Part: Fuel pump, CD-FP-02 — stock: 0 (Broekdam).
[1 calls, 147 input tokens, 63 output tokens]



> Bus RR14-2022-0210, error F-RR-120 at the last stop.
Likely cause: Auxiliary battery pack issue; door controller is not responding.
First action: Charge the auxiliary battery pack; replace it if needed.
Part: Auxiliary battery pack RR-BP-03 — stock 5 at Veldhaven.
[1 calls, 140 input tokens, 52 output tokens]

> The ramp on MM09-2025-2311 will not come out. No code on the display.
No fleet number or error code : sent to second line.
[0 calls, 0 input tokens, 0 output tokens]



The agent gets the four tools and the same instruction. It decides what to
look up, and how often.

In [6]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=[bus_info, part_stock, parts_for_model, search_manuals],
                     system_prompt=f"You are the Veenhaven Bus service desk. {NOTE} Use the tools.")

for report in REPORTS:
    meter = Meter()
    messages = agent.invoke({"messages": [{"role": "user", "content": report}]}, {"callbacks": [meter]})["messages"]
    runs[report]["agent"] = meter
    print(f"> {report}\ntools: {[m.name for m in messages if m.type == 'tool']}")
    print(f"{messages[-1].content}\n[{meter}]\n")

> CE12-2021-0117 : F-CE-110 on the afternoon route.
tools: ['bus_info', 'search_manuals', 'parts_for_model']
Likely cause: F-CE-110 indicates traction system over-temperature, possibly due to a weak cooling fan.
First action: Check the traction cooling fan; replace it if airflow is below 80%.
Part: CE-CF-02 traction cooling fan — 6 in stock at Rietpoort.
[4 calls, 2,079 input tokens, 148 output tokens]



> CD12-2024-0098 shows F-CD-202 again, pressure low.
tools: ['bus_info', 'search_manuals', 'parts_for_model']
Likely cause: F-CD-202 indicates low fuel-pump pressure.
First action: Withdraw the bus from service and call the diesel specialist; pump replacement requires a workshop bay.
Part: Fuel pump (CD-FP-02) — 0 in stock.
[3 calls, 1,308 input tokens, 154 output tokens]



> Bus RR14-2022-0210, error F-RR-120 at the last stop.
tools: ['bus_info', 'search_manuals', 'search_manuals', 'part_stock']
Likely cause: Door controller not responding, possibly due to a depleted auxiliary battery pack.
First action: Charge or replace the auxiliary battery pack.
Part: RR-BP-03 auxiliary battery pack — 5 in stock at Veldhaven.
[5 calls, 2,870 input tokens, 166 output tokens]



> The ramp on MM09-2025-2311 will not come out. No code on the display.
tools: ['bus_info', 'search_manuals', 'parts_for_model', 'search_manuals', 'parts_for_model']
Likely cause: Ramp motor stall or failure; no displayed code means this is unconfirmed.
First action: Run the ramp test in the workshop; isolate the bus before any component work.
Part: Wheelchair ramp motor, MM-RM-01 — 3 in stock (Zandveen).
[4 calls, 3,164 input tokens, 205 output tokens]



On the three reports with a code, both usually reach the same part ; the
workflow does it with one model call. On the ramp report the workflow
escalates, because its path has no step for a report without a code. The
agent can search the manuals for the symptom instead. Whether it finds the
right procedure varies between runs, and the cost is several calls.

In [7]:
print(f"{'report':40} {'workflow':>16} {'agent':>16}")
for report, r in runs.items():
    w, a = r["workflow"], r["agent"]
    print(f"{report[:40]:40} {w.calls:>3} calls {w.input:>6,} {a.calls:>3} calls {a.input:>6,}")

report                                           workflow            agent
CE12-2021-0117 : F-CE-110 on the afterno   1 calls    145   4 calls  2,079
CD12-2024-0098 shows F-CD-202 again, pre   1 calls    147   3 calls  1,308
Bus RR14-2022-0210, error F-RR-120 at th   1 calls    140   5 calls  2,870
The ramp on MM09-2025-2311 will not come   0 calls      0   4 calls  3,164


## Things to try

Optional.

- Add a node to the workflow that calls the agent for reports without a
  code : a workflow with one agentic step
- Write a report with a typo in the error code and run both